# Certified Counterfactual Generation on MNIST (Latent Space)

This notebook demonstrates the **CertifiedAtlas** API operating in the **latent space** of a convolutional autoencoder.

**Key idea**: Instead of building certified polytopes in the full 784D input space, we:
1. **Encode** training samples into a compact 32D latent space using a trained autoencoder
2. **Build the atlas** in latent space using a composite model: `decoder → classifier`
3. **Find counterfactuals** in the 32D latent space (dramatically cheaper QP solves)
4. **Decode** the results back to image space for visualization

Advantages over input-space atlas:
- QP optimization in 32D vs 784D — orders of magnitude faster
- Latent space captures meaningful structure — counterfactuals follow learned manifold
- Decoded images are more natural-looking (constrained to autoencoder's image manifold)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import TensorDataset
from preimage_sampling.models import MNISTClassifier
from preimage_sampling.models import ConvAutoencoder
from preimage_sampling.atlas import CertifiedAtlas

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2 – Load Data & Models

We load the MNIST test set, the pre-trained CNN classifier, and the pre-trained convolutional autoencoder. We then:
1. Create a **composite model** (`decoder → classifier`) that maps latent vectors to class logits
2. **Encode** all samples to the 32D latent space

In [ ]:
test_transform = transforms.Compose([
    transforms.ToTensor(),
])
full_test = datasets.MNIST(root='./data', train=False, download=True, transform=test_transform)

# Select a balanced subset (N_PER_CLASS samples per digit)
N_PER_CLASS = 1
targets = full_test.targets.numpy()

indices = []
for c in range(10):
    class_idx = np.where(targets == c)[0][:N_PER_CLASS]
    indices.extend(class_idx)

# Materialize transformed images into a TensorDataset
images = torch.stack([full_test[i][0] for i in indices])
labels = torch.tensor([full_test[i][1] for i in indices])

print(f'Dataset: {len(images)} samples ({N_PER_CLASS} per class), shape: {images.shape}')

# Load pre-trained CNN classifier
model = MNISTClassifier(num_classes=10).to(device)
model.load_state_dict(torch.load('data/Trained Models/mnist_classifier.pth',
                                  map_location=device, weights_only=True))
model.eval()

# Load pre-trained MNIST AutoEncoder
autoencoder = ConvAutoencoder().to(device)
autoencoder.load_state_dict(torch.load('data/Trained Models/conv_ae.pth',
                                      map_location=device, weights_only=True))
autoencoder.eval()

# Verify classifier accuracy
with torch.no_grad():
    preds = model(images.to(device)).argmax(dim=1).cpu()
    acc = (preds == labels).float().mean()
print(f'Classifier accuracy on subset: {acc*100:.1f}%')

# --- Composite model: decoder → classifier ---
class DecoderClassifier(nn.Module):
    """Maps latent vectors (32D) → class logits (10D) via decoder then classifier."""
    def __init__(self, decoder, classifier):
        super().__init__()
        self.decoder = decoder
        self.classifier = classifier

    def forward(self, z):
        x_reconstructed = self.decoder(z)       # (batch, 1, 28, 28)
        return self.classifier(x_reconstructed)  # (batch, 10)

composite_model = DecoderClassifier(autoencoder.decoder, model).to(device)
composite_model.eval()

# --- Encode all data to latent space ---
with torch.no_grad():
    latent_vectors = autoencoder.encoder(images.to(device)).cpu()  # (1000, 32)

print(f'Latent space dimension: {latent_vectors.shape[1]}')

# Verify composite model accuracy (encoder → decoder → classifier)
with torch.no_grad():
    composite_preds = composite_model(latent_vectors.to(device)).argmax(dim=1).cpu()
    composite_acc = (composite_preds == labels).float().mean()
print(f'Composite model accuracy (encode→decode→classify): {composite_acc*100:.1f}%')

# Create latent dataset for atlas
latent_dataset = TensorDataset(latent_vectors, labels)

# Helper: decode latent vector to displayable image
def decode_to_image(z_flat):
    """Decode a flat latent vector (32D) to a 28x28 image."""
    with torch.no_grad():
        z = torch.tensor(z_flat, dtype=torch.float32).unsqueeze(0).to(device)
        img = autoencoder.decoder(z).cpu().numpy()[0, 0]  # (28, 28)
    return np.clip(img, 0, 1)

## 3 – Build Certified Atlas in Latent Space

The atlas is built using the **composite model** (`decoder → classifier`) with 32D latent vectors as input. LiRPA computes certified linear bounds in the latent space:

$$A_i \mathbf{z} + \mathbf{b}_i \leq f_t(\text{decoder}(\mathbf{z})) - f_j(\text{decoder}(\mathbf{z})) \quad \forall j \neq t$$

Since we're in 32D instead of 784D, QP solves are dramatically faster.

In [ ]:
# Build atlas in latent space using composite model (decoder → classifier)
# cnn=False because input is flat 32D latent vectors
# NOTE: must use L∞ norm because auto_LiRPA only supports IBP through ConvTranspose2d with L∞
atlas = CertifiedAtlas(composite_model, latent_dataset, device, cnn=False, solver_maxiter=3000)

atlas.build(
    eps=0.1,
    norm=np.inf,
    batch_size=20,
    verbose=True
)

In [ ]:
print(atlas.summary())

In [ ]:
# Check polytope feasibility across all classes
print('Polytope feasibility analysis:\n')

total_feasible = 0
total = 0
for label in range(10):
    bd = atlas.bounds[label]
    n = len(bd['X'])
    feasible = 0
    for i in range(n):
        margins = bd['lA'][i] @ bd['X'][i] + bd['lbias'][i]
        if margins.min() >= -1e-6:
            feasible += 1
    total_feasible += feasible
    total += n
    print(f'  Digit {label}: {feasible}/{n} polytopes feasible ({100*feasible/n:.0f}%)')

print(f'\nTotal: {total_feasible}/{total} ({100*total_feasible/total:.0f}%)')

In [ ]:
# Polytope size distribution via Chebyshev radius
# In 784D, volume is intractable. The Chebyshev radius r = min_i (a_i^T c + b_i) / ||a_i||_2
# is the radius of the largest L2-ball centered at c that fits inside the halfspace constraints.
# The box/ball constraint further caps r at eps.

all_radii = {}
for label in range(10):
    bd = atlas.bounds[label]
    radii = []
    for i in range(len(bd['X'])):
        margins = bd['lA'][i] @ bd['X'][i] + bd['lbias'][i]
        if margins.min() < -1e-6:
            continue  # infeasible
        row_norms = np.linalg.norm(bd['lA'][i], axis=1, ord=2)
        row_norms = np.maximum(row_norms, 1e-12)
        r = np.min(margins / row_norms)
        r = min(r, atlas.eps)  # capped by the ball/box constraint
        radii.append(r)
    all_radii[label] = np.array(radii)

cmap = plt.colormaps.get_cmap('tab10')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overlaid histogram
ax = axes[0]
for label in range(10):
    if len(all_radii[label]) > 0:
        ax.hist(all_radii[label], bins=30, alpha=0.4, color=cmap(label),
                label=f'{label} (n={len(all_radii[label])})')
ax.set_xlabel('Chebyshev radius')
ax.set_ylabel('Count')
ax.set_title('Distribution of Polytope Chebyshev Radii by Digit')
ax.legend(fontsize=7, ncol=2)
ax.set_xlim(left=0)

# Box plot
ax = axes[1]
data = [all_radii[l] for l in range(10)]
bp = ax.boxplot(
    data,
    tick_labels=[str(l) for l in range(10)],
    patch_artist=True
)
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor(cmap(i))
    patch.set_alpha(0.6)
ax.set_xlabel('Digit')
ax.set_ylabel('Chebyshev radius')
ax.set_title('Chebyshev Radius per Digit Class')

plt.tight_layout()
plt.show()

# Summary
all_flat = np.concatenate(list(all_radii.values()))
total = sum(len(r) for r in all_radii.values())
print(f'Feasible polytopes: {total}/10000')
print(f'Chebyshev radius: mean={all_flat.mean():.4f}, median={np.median(all_flat):.4f}, '
      f'min={all_flat.min():.4f}, max={all_flat.max():.4f}')
print(f'Polytopes at max radius (eps={atlas.eps}): '
      f'{np.sum(np.isclose(all_flat, atlas.eps, atol=1e-6))}/{total}')

## 4 – Visualize Dataset Samples (Decoded from Latent Space)

We decode latent vectors back to image space to visualize the training samples used by the atlas.

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))
for digit in range(10):
    Z_digit = atlas.bounds[digit]['X']
    for row in range(2):
        ax = axes[row, digit]
        ax.imshow(decode_to_image(Z_digit[row]), cmap='gray', vmin=0, vmax=1)
        ax.axis('off')
        if row == 0:
            ax.set_title(str(digit), fontsize=12)

fig.suptitle('MNIST Samples (decoded from latent space, 2 per class)', fontsize=14)
plt.tight_layout()
plt.show()

## 5 – Generate Counterfactuals in Latent Space

For selected query digits, we find the closest certified counterfactual in the **latent space**. The query point is encoded to 32D, the counterfactual search operates entirely in latent space, and results are decoded back to images for visualization.

Given a query latent vector $\mathbf{z}_0 = \text{encoder}(\mathbf{x}_0)$, we solve:

$$\min_{\mathbf{z}} \;\|\mathbf{z} - \mathbf{z}_0\|_2^2 \qquad \text{s.t.} \quad A_i \mathbf{z} + \mathbf{b}_i \geq \mathbf{0}, \quad \|\mathbf{z} - \mathbf{c}_i\|_p \leq \varepsilon$$

The counterfactual image is then $\mathbf{x}_{cf} = \text{decoder}(\mathbf{z}_{cf})$.

In [ ]:
source_class = 1
n_queries = 3
# Query points are latent vectors from the atlas
query_points = atlas.bounds[source_class]['X'][:n_queries]

print(f'Generating counterfactuals for {n_queries} queries from digit {source_class}')
print(f'(operating in {query_points.shape[1]}D latent space)\n')

all_results = {}
for target_class in range(10):
    if target_class == source_class:
        continue

    results = []
    for i, z0 in enumerate(query_points):
        result = atlas.find_counterfactual(z0, target_class, method='bvh')
        results.append(result)

    all_results[target_class] = results
    n_ok = sum(r.success for r in results)
    avg_dist = np.mean([r.distance for r in results if r.success]) if n_ok > 0 else float('inf')
    avg_qp = np.mean([r.n_qp_solved for r in results])
    print(f'  -> Digit {target_class}: {n_ok}/{n_queries} success, '
          f'avg latent dist={avg_dist:.4f}, avg QPs={avg_qp:.1f}')

## 6 – Verify Counterfactuals

All counterfactuals lie inside certified polytopes, so they are **guaranteed** to be classified correctly.

In [ ]:
print('Verifying counterfactual validity:\n')

all_valid = True
for target_class, results in all_results.items():
    for i, result in enumerate(results):
        if not result.success:
            all_valid = False
            continue
        verification = atlas.verify_counterfactual(result.x_cf, target_class)
        if not verification['valid']:
            print(f'  INVALID: query {i} -> digit {target_class}, '
                  f'pred={verification["predicted"]}')
            all_valid = False

print(f'All counterfactuals valid: {all_valid}')

## 7 – Counterfactual Interpolation Paths (in Latent Space)

For each target class, we interpolate in the **latent space**: $\mathbf{z}(\alpha) = (1-\alpha)\,\mathbf{z}_0 + \alpha\,\mathbf{z}_{cf}$ and decode each interpolant to image space. The prediction at each step is computed by the full pipeline (decode then classify).

In [ ]:
query_idx = 0
z_orig = query_points[query_idx]

target_classes = sorted(all_results.keys())
n_targets = len(target_classes)

# Interpolation steps
alphas = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
n_steps = len(alphas)
n_rows = n_steps + 1  # +1 for difference map

fig, axes = plt.subplots(n_rows, n_targets, figsize=(2 * n_targets, 1.8 * n_rows))

# Row labels
row_labels = [f'$\\alpha$={a:.1f}' for a in alphas] + ['Difference']
row_labels[0] = 'Original'
row_labels[-2] = 'CF'
for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=10, rotation=0, labelpad=55, va='center')

# Decode original for difference computation
img_orig = decode_to_image(z_orig)

for col, target_class in enumerate(target_classes):
    result = all_results[target_class][query_idx]

    if not result.success:
        axes[0, col].set_title(f'{target_class}\n(failed)', fontsize=10)
        for row in range(n_rows):
            axes[row, col].axis('off')
        continue

    z_cf = result.x_cf
    axes[0, col].set_title(f'{target_class}  (d={result.distance:.3f})', fontsize=10)

    # Interpolation rows
    for row, alpha in enumerate(alphas):
        ax = axes[row, col]
        z_interp = (1 - alpha) * z_orig + alpha * z_cf
        img_interp = decode_to_image(z_interp)
        ax.imshow(img_interp, cmap='gray', vmin=0, vmax=1)
        ax.set_xticks([])
        ax.set_yticks([])

        # Show model prediction (via composite model)
        with torch.no_grad():
            z_t = torch.tensor(z_interp, dtype=torch.float32).unsqueeze(0).to(device)
            pred = composite_model(z_t).argmax(dim=1).item()
        color = 'lime' if pred == target_class else ('white' if pred == source_class else 'orange')
        ax.text(0.97, 0.03, str(pred), fontsize=8, color=color, fontweight='bold',
                ha='right', va='bottom', transform=ax.transAxes,
                bbox=dict(boxstyle='round,pad=0.15', fc='black', alpha=0.7))

    # Difference row (in image space)
    img_cf = decode_to_image(z_cf)
    diff = img_cf - img_orig
    vmax = max(abs(diff.min()), abs(diff.max()))
    ax = axes[-1, col]
    if vmax > 0:
        ax.imshow(diff, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    else:
        ax.imshow(diff, cmap='RdBu_r')
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle(f'Latent-space interpolation: digit {source_class} to each target', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 8 – δ-Robust Counterfactuals in Latent Space

Same δ-robustness mechanism, but now applied in the 32D latent space. The erosion guarantees that the entire ball $\mathcal{B}_q(\mathbf{z}_{cf}, \delta)$ in latent space lies inside the certified polytope.

In [ ]:
z_query = atlas.bounds[3]['X'][0]
target = 8

print(f'Query: digit 3, target: digit {target}')
print(f'Atlas norm: L{atlas.norm}, eps={atlas.eps}')
print(f'Latent dim: {len(z_query)}\n')

delta_values = [0.0, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2]

print(f'--- Robustness norm = L{atlas.norm} (same as atlas) ---')
for delta in delta_values:
    result = atlas.find_counterfactual(z_query, target, method='bvh', delta=delta)
    if result.success:
        v = atlas.verify_counterfactual(result.x_cf, target)
        print(f'  delta={delta:.3f}: latent_dist={result.distance:.4f}, QPs={result.n_qp_solved}, valid={v["valid"]}')
    else:
        print(f'  delta={delta:.3f}: FAILED (eroded polytope empty)')

print()
print('--- Robustness norm = L∞ (cross-norm) ---')
for delta in delta_values:
    result = atlas.find_counterfactual(z_query, target, method='bvh',
                                       delta=delta, robust_norm=np.inf)
    if result.success:
        v = atlas.verify_counterfactual(result.x_cf, target)
        print(f'  delta={delta:.3f}: latent_dist={result.distance:.4f}, QPs={result.n_qp_solved}, valid={v["valid"]}')
    else:
        print(f'  delta={delta:.3f}: FAILED (eroded polytope empty)')

In [ ]:
# Visualize the effect of increasing delta on a counterfactual (decoded to image space)
z_query_vis = atlas.bounds[1]['X'][0]
target = 7

fig, axes = plt.subplots(2, len(delta_values) + 1, figsize=(2.5 * (len(delta_values) + 1), 5))

# Show original (decoded)
axes[0, 0].imshow(decode_to_image(z_query_vis), cmap='gray', vmin=0, vmax=1)
axes[0, 0].set_title(f'Original\n(digit 1)', fontsize=10)
axes[1, 0].axis('off')
for row in range(2):
    axes[row, 0].set_xticks([])
    axes[row, 0].set_yticks([])

img_orig_vis = decode_to_image(z_query_vis)

for col, delta in enumerate(delta_values, 1):
    result = atlas.find_counterfactual(z_query_vis, target, method='bvh', delta=delta)
    for row in range(2):
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])

    if result.success:
        img_cf = decode_to_image(result.x_cf)
        axes[0, col].imshow(img_cf, cmap='gray', vmin=0, vmax=1)
        axes[0, col].set_title(f'δ={delta}\nd={result.distance:.3f}', fontsize=10)

        diff = img_cf - img_orig_vis
        vmax = max(abs(diff.min()), abs(diff.max()))
        if vmax > 0:
            axes[1, col].imshow(diff, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        else:
            axes[1, col].imshow(diff, cmap='RdBu_r')
        axes[1, col].set_title('Difference', fontsize=9)
    else:
        axes[0, col].set_title(f'δ={delta}\n(failed)', fontsize=10)
        axes[0, col].axis('off')
        axes[1, col].axis('off')

fig.suptitle(f'Effect of robustness radius δ in latent space (digit 1 → digit {target})', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 9 – Benchmark: BVH vs k-NN Search (Latent Space)

In 32D latent space, each QP solve is much cheaper than in 784D. Let's see how BVH pruning compares.

In [ ]:
import time

n_test = 5
test_queries = atlas.bounds[0]['X'][:n_test]
target = 9
n_polytopes = len(atlas.bounds[target]['X'])

print(f'Benchmarking {n_test} queries (digit 0 -> digit {target})')
print(f'Total polytopes in target class: {n_polytopes}')
print('=' * 60)

bvh_qps, knn_qps = [], []
bvh_times, knn_times = [], []

for i, x0 in enumerate(test_queries):
    # BVH
    t0 = time.time()
    result_bvh = atlas.find_counterfactual(x0, target, method='bvh')
    t_bvh = time.time() - t0

    # k-NN (k = all polytopes)
    t0 = time.time()
    result_knn = atlas.find_counterfactual(x0, target, method='knn', k=n_polytopes)
    t_knn = time.time() - t0

    bvh_qps.append(result_bvh.n_qp_solved)
    knn_qps.append(result_knn.n_qp_solved)
    bvh_times.append(t_bvh)
    knn_times.append(t_knn)

    match = np.isclose(result_bvh.distance, result_knn.distance, atol=1e-3)
    print(f'Query {i}: BVH {result_bvh.n_qp_solved:3d} QPs ({t_bvh:5.1f}s) | '
          f'k-NN {result_knn.n_qp_solved:3d} QPs ({t_knn:5.1f}s) | '
          f'dist={result_bvh.distance:.2f} {"✓" if match else "✗"}')

print('=' * 60)
print(f'Avg QPs:  BVH = {np.mean(bvh_qps):.1f},  k-NN = {np.mean(knn_qps):.1f}  '
      f'({np.mean(knn_qps)/np.mean(bvh_qps):.1f}x fewer with BVH)')
print(f'Avg time: BVH = {np.mean(bvh_times):.1f}s,  k-NN = {np.mean(knn_times):.1f}s  '
      f'({np.mean(knn_times)/np.mean(bvh_times):.1f}x speedup with BVH)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# QPs comparison
ax = axes[0]
x_pos = np.arange(n_test)
width = 0.35
ax.bar(x_pos - width/2, knn_qps, width, label='k-NN (exhaustive)', color='coral')
ax.bar(x_pos + width/2, bvh_qps, width, label='BVH (branch & bound)', color='steelblue')
ax.set_xlabel('Query index')
ax.set_ylabel('QP problems solved')
ax.set_title('QP Problems Solved per Query')
ax.legend()

# Time comparison
ax = axes[1]
ax.bar(x_pos - width/2, knn_times, width, label='k-NN (exhaustive)', color='coral')
ax.bar(x_pos + width/2, bvh_times, width, label='BVH (branch & bound)', color='steelblue')
ax.set_xlabel('Query index')
ax.set_ylabel('Time (seconds)')
ax.set_title('Wall-Clock Time per Query')
ax.legend()

plt.tight_layout()
plt.show()

## 10 – Counterfactual Quality Metrics

We compute metrics both in **latent space** (where the atlas operates) and in **image space** (decoded).

In [ ]:
from sklearn.neighbors import LocalOutlierFactor
import pandas as pd

# Generate counterfactuals for several source/target pairs and compute metrics
source_digits = [0, 1, 3, 7]
n_queries_per = 5

rows = []
for source_class in source_digits:
    queries = atlas.bounds[source_class]['X'][:n_queries_per]

    for target_class in range(10):
        if target_class == source_class:
            continue

        # LOF in latent space
        target_data = atlas.bounds[target_class]['X']
        n_neighbors = min(10, len(target_data) - 1)
        lof = LocalOutlierFactor(n_neighbors=n_neighbors, novelty=True)
        lof.fit(target_data)

        for i, z_q in enumerate(queries):
            result = atlas.find_counterfactual(z_q, target_class, method='bvh')
            if not result.success:
                continue

            z_cf = result.x_cf
            z_diff = z_cf - z_q

            # Decode both to image space for pixel-level metrics
            img_orig = decode_to_image(z_q)
            img_cf = decode_to_image(z_cf)
            img_diff = img_cf - img_orig

            rows.append({
                'source': source_class,
                'target': target_class,
                'query_idx': i,
                # Latent space metrics
                'latent_l2_distance': np.linalg.norm(z_diff),
                'latent_linf_distance': np.max(np.abs(z_diff)),
                'lof_score': -lof.score_samples(z_cf.reshape(1, -1))[0],
                # Image space metrics (decoded)
                'image_l2_distance': np.linalg.norm(img_diff),
                'image_linf_distance': np.max(np.abs(img_diff)),
                'n_pixels_changed': int(np.sum(np.abs(img_diff) > 0.01)),
                'sparsity': np.sum(np.abs(img_diff) > 0.01) / 784,
                'n_qp_solved': result.n_qp_solved
            })

metrics_df = pd.DataFrame(rows)
print(f'Generated {len(metrics_df)} counterfactuals\n')
metrics_df.head(10)

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Latent L2 distance by target class
sns.boxplot(ax=axes[0, 0], x='target', y='latent_l2_distance', data=metrics_df,
            hue='target', palette='Set2', legend=False)
axes[0, 0].set_title('Latent L2 Distance by Target Digit')
axes[0, 0].set_xlabel('Target digit')

# Image L2 distance (decoded)
sns.boxplot(ax=axes[0, 1], x='target', y='image_l2_distance', data=metrics_df,
            hue='target', palette='viridis', legend=False)
axes[0, 1].set_title('Image L2 Distance (Decoded) by Target Digit')
axes[0, 1].set_xlabel('Target digit')

# LOF plausibility (in latent space)
sns.violinplot(ax=axes[1, 0], x='target', y='lof_score', data=metrics_df,
               hue='target', palette='pastel', legend=False, inner='quart')
axes[1, 0].axhline(1.0, color='red', linestyle='--', alpha=0.6, label='LOF=1 (inlier)')
axes[1, 0].set_title('Plausibility (LOF Score in Latent Space)')
axes[1, 0].set_xlabel('Target digit')
axes[1, 0].legend()

# QPs solved (efficiency)
sns.barplot(ax=axes[1, 1], x='target', y='n_qp_solved', data=metrics_df,
            hue='target', palette='muted', legend=False, estimator='mean')
axes[1, 1].set_title('Avg QP Problems Solved (BVH)')
axes[1, 1].set_xlabel('Target digit')

plt.tight_layout()
plt.show()

## Summary

We demonstrated certified counterfactual generation in the **latent space** of a convolutional autoencoder:

```python
# Composite model: decoder → classifier
composite_model = DecoderClassifier(autoencoder.decoder, classifier)

# Encode data to latent space
latent_vectors = autoencoder.encoder(images)
latent_dataset = TensorDataset(latent_vectors, labels)

# Build atlas in 32D latent space (offline)
atlas = CertifiedAtlas(composite_model, latent_dataset, device, cnn=False)
atlas.build(eps=0.5, norm=2)

# Find counterfactual in latent space (online)
z_query = autoencoder.encoder(x_query)
result = atlas.find_counterfactual(z_query, target_class=3)

# Decode back to image space
x_cf = autoencoder.decoder(result.x_cf)
```

Key observations:
- **32D vs 784D**: QP solves are dramatically faster in latent space
- **Manifold-constrained**: Decoded counterfactuals follow the autoencoder's learned image manifold
- **Certified validity**: All counterfactuals are guaranteed to be classified as the target digit (through the decoder→classifier pipeline)
- **Same δ-robustness**: Robustness guarantees transfer to latent space perturbations